# DUK leg kinematics

This notebook uses the canonical DUK robot frame directly. **Only A is fixed at (0, 0).** D, B, C, H and O are derived from lengths, directed angles and physical left/right topology.

- +Ox points right, +Oy points up, counter-clockwise is positive.
- D→A = -30°, therefore A→D = +150°.
- `ab_angle` is the absolute direction of A→B measured from +Ox.
- B, C and O are fixed material points of one rigid carrier.
- No mirror, solver-frame rotation or display correction is applied after solving.


In [ ]:
from pathlib import Path
import math

from duk_core.modeling import load_leg_geometry_json, solve_leg_pose, leg_jacobian, sample_leg_trajectory
from duk_core.visualization import draw_leg_linkage, draw_point_trajectories, plot_point_coordinates, plot_point_jacobian

def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'robot/config/physical/duk_leg_geometry.json').exists():
            return candidate
    raise FileNotFoundError('Run the notebook from a DUK repository checkout')

repo_root = find_repo_root()
geometry = load_leg_geometry_json(repo_root / 'robot/config/physical/duk_leg_geometry.json')
print(f'DA = {math.degrees(geometry.da_angle_rad):+.1f}°')
print(f'AD = {math.degrees(geometry.ad_angle_rad):+.1f}°')
print('D derived from AD and angle =', geometry.d_point, 'mm')
geometry


In [ ]:
lower, upper = geometry.geometric_ab_angle_limits_rad()
print(f'BO = {geometry.bo_mm:.3f} mm')
print(f'CO = {geometry.co_mm:.3f} mm')
print(f'physical A→B angle range = {math.degrees(lower):.3f} .. {math.degrees(upper):.3f}°')
print('The endpoints are geometric singular boundaries, not operating limits.')


In [ ]:
ab_angle_deg = 75.0
pose = solve_leg_pose(geometry, math.radians(ab_angle_deg))
jac = leg_jacobian(geometry, pose)
print(f'A→B angle = {ab_angle_deg:.2f}° from +Ox')
for name in ('A', 'B', 'C', 'D', 'H', 'O'):
    print(name, '=', pose.point(name), 'mm')
print('dO/d(ab_angle) =', jac.d_o, 'mm/rad')
draw_leg_linkage(pose, geometry).set_title(f'DUK leg — A→B={ab_angle_deg:.1f}°')


In [ ]:
margin = math.radians(2.0)
poses = sample_leg_trajectory(geometry, lower + margin, upper - margin, 180)
draw_point_trajectories(poses).set_title('B/C/H/O trajectories — canonical robot frame')


In [ ]:
plot_point_coordinates(poses, 'O').set_title('Wheel-center coordinates — canonical robot frame')


In [ ]:
plot_point_jacobian(geometry, poses, 'O').set_title('Wheel-center Jacobian — canonical robot frame')
